# FSAKE: Few-Shot Graph Learning via Adaptive Neighbor Class Knowledge Embedding
## Google Colab Step-by-Step Guide

This notebook provides a complete, reproducible workflow to run the **FSAKE** model in Google Colab.

**Paper**: FSAKE: Few-shot graph learning via adaptive neighbor class knowledge embedding  
**GitHub**: https://github.com/fhqxa/FSAKE  
**Reference**: Expert Systems with Applications (ScienceDirect)

---

### Required Environment (from official README)

- Python 3.8
- PyTorch 1.10.0
- pytorch_geometric 2.0.1
- opencv-python 3.4.10.35
- tensorboardx 2.6.2.2

### Before you start
- Enable GPU: `Runtime -> Change runtime type -> Hardware accelerator -> GPU -> Save`
- FSAKE datasets are **pickle files** (not raw image folders) — see Step 3 for exact structure

In [ ]:
# Verify GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Go to Runtime → Change runtime type → GPU")

## Step 1: Clone the Repository with Fixed FSAKE

This clones the main repository which includes the `fsake_repo_clone` submodule with all pickle format fixes (binary mode, encoding fallbacks, split format detection, and GPU device handling corrections).

In [ ]:
import os
import shutil

# Clone the main repository with submodules (includes fsake_repo_clone with all fixes)
if not os.path.exists('/content/few_shot_article'):
    !git clone --recursive https://github.com/alirezakavianifar/few_shot_article.git /content/few_shot_article
    print("Repository cloned successfully with submodules!")
else:
    print("Repository already cloned.")

# Copy the fixed fsake_repo_clone to FSAKE for training
if os.path.exists('/content/few_shot_article/fsake_repo_clone'):
    if os.path.exists('/content/FSAKE'):
        shutil.rmtree('/content/FSAKE')
    shutil.copytree('/content/few_shot_article/fsake_repo_clone', '/content/FSAKE')
    print("Copied fixed fsake_repo_clone to /content/FSAKE")
else:
    # Fallback: clone original FSAKE if submodule not available
    !git clone https://github.com/fhqxa/FSAKE.git /content/FSAKE
    print("Cloned original FSAKE (submodule not available)")

%cd /content/FSAKE
!ls

In [ ]:
# Verify that the fixed version was copied correctly
# The fsake_repo_clone includes all necessary patches:
# ✓ Windows backslash path fix (mini-imagenet path)
# ✓ Pickle binary mode fixes (all loaders)
# ✓ Split format detection for pickle data
# ✓ Byte-key conversion for class dictionaries
# ✓ Encoding fallback chain (default → latin1 → bytes)
# ✓ GPU device mismatch fix in one_hot_encode()

import os

verification_checks = []

# Check 1: Verify data.py exists and has fixes
data_py = '/content/FSAKE/data.py'
if os.path.exists(data_py):
    with open(data_py, 'r') as f:
        content = f.read()
    
    checks = [
        ('os.path.join(self.root, \'mini-imagenet\', \'compacted_datasets\'', 'Mini-ImageNet path fix'),
        ('with open(labels_name, \'rb\')', 'Binary mode fix'),
        ('encoding=\'latin1\'', 'Encoding fallback'),
        ('catname2label', 'Split format detection'),
        ('class_idx.cpu()', 'GPU device fix'),
    ]
    
    for pattern, name in checks:
        if pattern in content:
            verification_checks.append(f"✓ {name}")
        else:
            verification_checks.append(f"✗ {name} - NOT FOUND")
    
    print("Verification Results:")
    for check in verification_checks:
        print(f"  {check}")
else:
    print(f"ERROR: data.py not found at {data_py}")

print("\nAll fixes from fsake_repo_clone have been applied!")

## Step 2: Verify Fixes and Install Dependencies

The cloned repo now includes all necessary fixes. This step verifies them and installs PyTorch + PyG.

In [ ]:
# The FSAKE repo does NOT have a requirements.txt.
# Install PyTorch (latest stable with CUDA 12.1 — compatible with Python 3.12 and Colab GPU)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch, subprocess, sys

# Build the pyg wheel URL from the installed torch version
torch_ver = torch.__version__.split('+')[0]          # e.g. "2.3.0"
cuda_tag  = 'cu' + torch.version.cuda.replace('.','') # e.g. "cu121"
pyg_url   = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print(f"PyG wheel index: {pyg_url}")

# Install pytorch_geometric (latest — no pre-built wheels exist for Python 3.12 + old 2.0.1)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric'], check=True)

# Install optional C++ extensions (scatter, sparse, cluster, spline-conv).
# These fall back gracefully to slower pure-Python ops if wheels aren't available.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
     '-f', pyg_url],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("Pre-built wheels not found for this env — installing source fallback...")
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv'],
                   check=False)
else:
    print("PyG extensions installed successfully.")

# Remaining dependencies (unpinned — works with Python 3.12)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy', 'scipy', 'scikit-learn', 'tqdm',
                'opencv-python', 'tensorboardx'], check=True)
print("All dependencies installed.")

## Step 3: Prepare Datasets (Critical Step)

The FSAKE code expects all datasets as **pre-processed pickle files** under `/content/FSAKE/dataset/`.  
The repo contains **no download scripts** — you must supply the files yourself.

> **Bug already patched above (Step 1 patch cell):** The original `data.py` uses a Windows-style  
> backslash (`'mini-imagenet\\compacted_datasets'`) which is invalid on Linux/Colab. The patch cell  
> after the clone step fixes this automatically.

### Where to get the pickle files

| Dataset arg | Folder inside `dataset/` | Files needed | Source repo |
|-------------|--------------------------|-------------|-------------|
| `mini` | `mini-imagenet/compacted_datasets/` | `mini_imagenet_train.pickle`, `_val.pickle`, `_test.pickle` | [few-shot-gnn (vgsatorras)](https://github.com/vgsatorras/few-shot-gnn) |
| `tiered` | `tiered-imagenet/` | `train_labels.pkl` + `train_images.npz` (+ val/test) | [Ren et al.](https://github.com/renmengye/few-shot-ssl-public) |
| `cub` | `cub-200-2011/` | `cub200_train.pickle`, `_val.pickle`, `_test.pickle` | FEAT / DeepEMD repos |
| `cifar` | `cifar_fs/` | `train.pickle`, `val.pickle`, `test.pickle` | DeepEMD / FEAT repos |

### Option A (Recommended): Mount Google Drive and copy pre-downloaded pickles


In [ ]:

# Option A: Mount Google Drive and link pre-downloaded pickles
# If this succeeds, Option B (auto-download) will be skipped automatically.

OPTION_A_SUCCESS = False

try:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust to the folder in your Drive that holds the dataset sub-folders
    DRIVE_DATASET_ROOT = '/content/drive/MyDrive/FSAKE_datasets'

    import os
    LOCAL_DATASET_ROOT = '/content/FSAKE/dataset'

    if not os.path.isdir(DRIVE_DATASET_ROOT):
        raise FileNotFoundError(f"Drive folder not found: {DRIVE_DATASET_ROOT}\n"
                                f"Create it or update DRIVE_DATASET_ROOT above.")

    os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)

    # Symlink drive folder so FSAKE finds data at dataset/
    if os.path.islink(LOCAL_DATASET_ROOT):
        os.remove(LOCAL_DATASET_ROOT)           # replace stale symlink
    elif os.path.isdir(LOCAL_DATASET_ROOT) and not os.listdir(LOCAL_DATASET_ROOT):
        os.rmdir(LOCAL_DATASET_ROOT)            # remove empty dir before symlinking

    if not os.path.exists(LOCAL_DATASET_ROOT):
        os.symlink(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
        print(f"Symlink created: {DRIVE_DATASET_ROOT} -> {LOCAL_DATASET_ROOT}")
    else:
        print(f"Dataset directory already populated: {LOCAL_DATASET_ROOT}")

    # Quick sanity check: at least one expected pickle must exist
    sentinel = os.path.join(LOCAL_DATASET_ROOT,
                            'mini-imagenet', 'compacted_datasets',
                            'mini_imagenet_train.pickle')
    if os.path.exists(sentinel):
        OPTION_A_SUCCESS = True
        print("Option A succeeded — Google Drive data found. Option B will be skipped.")
    else:
        print("WARNING: Drive mounted but expected pickle files not found inside.")
        print(f"  Expected: {sentinel}")
        print("  Option B (auto-download) will run next.")

except Exception as e:
    print(f"Option A failed: {e}")
    print("Option B (auto-download via gdown) will run next.")


### Option B: Auto-download via gdown (real file IDs, no manual upload needed)

Files sourced from well-known public few-shot learning repos:

| Dataset | Source repo | Google Drive ID |
|---------|-------------|-----------------|
| miniImageNet | [MetaOptNet / Gidaris](https://github.com/kjunelee/MetaOptNet) | `12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D` |
| tieredImageNet | [MetaOptNet](https://github.com/kjunelee/MetaOptNet) | `1nVGCTd9ttULRXFezh4xILQ9lUkg0WZCG` |
| CIFAR-FS | [MetaOptNet / Bertinetto et al.](https://github.com/kjunelee/MetaOptNet) | `1GjGMI0q3bgcpcB_CjI40fX54WgLPuTpS` |
| CUB-200-2011 | [DeepEMD](https://github.com/icoz69/DeepEMD) | `1B8jmZin9teye7Lte9ZKsQ3lyMASbxune` |

> These are the same splits and image sizes (84×84) used by the FSAKE paper.  
> Run only the datasets you need — each download is 0.5–2 GB.


In [ ]:

# Option B: Auto-download via gdown
# Skipped automatically if Option A already populated the dataset directory.

if 'OPTION_A_SUCCESS' not in dir() or not OPTION_A_SUCCESS:
    import os, subprocess, sys, tarfile, zipfile, pickle, shutil, csv as csv_mod
    import numpy as np
    from PIL import Image as pil_image

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown

    DATASET_ROOT = '/content/FSAKE/dataset'
    os.makedirs(DATASET_ROOT, exist_ok=True)

    # ── helpers ──────────────────────────────────────────────────────────────

    def _extract(archive_path, extract_to):
        os.makedirs(extract_to, exist_ok=True)
        with open(archive_path, 'rb') as fh:
            magic = fh.read(4)
        if magic[:2] == b'PK':
            print(f"  Detected ZIP — extracting...")
            with zipfile.ZipFile(archive_path) as z:
                z.extractall(extract_to)
        elif magic[:2] == b'\x1f\x8b' or magic[:3] in (b'BZh', b'\xfd7z'):
            print(f"  Detected TAR — extracting...")
            with tarfile.open(archive_path) as t:
                t.extractall(extract_to, filter='data')
        else:
            try:
                with tarfile.open(archive_path) as t:
                    t.extractall(extract_to, filter='data')
                print(f"  Extracted as TAR.")
            except tarfile.ReadError:
                with zipfile.ZipFile(archive_path) as z:
                    z.extractall(extract_to)
                print(f"  Extracted as ZIP.")

    def gdrive_download(file_id, out_path):
        """Try 4 gdown API variants; raises RuntimeError if all fail."""
        approaches = [
            # 1. fuzzy=True parses /file/d/ URL
            lambda: gdown.download(
                f'https://drive.google.com/file/d/{file_id}/view',
                out_path, quiet=False, fuzzy=True),
            # 2. id= keyword with cookies
            lambda: gdown.download(id=file_id, output=out_path, quiet=False),
            # 3. use_cookies=False — direct endpoint, bypasses some quota/permission errors
            lambda: gdown.download(id=file_id, output=out_path,
                                   quiet=False, use_cookies=False),
            # 4. legacy URL
            lambda: gdown.download(
                f'https://drive.google.com/uc?export=download&id={file_id}',
                out_path, quiet=False),
        ]
        names = ['fuzzy-URL', 'id+cookies', 'id+no-cookies', 'legacy-URL']
        errors = []
        for name, fn in zip(names, approaches):
            try:
                fn()
                return
            except Exception as e:
                errors.append(f'[{name}] {e}')
                if os.path.exists(out_path):
                    os.remove(out_path)
        raise RuntimeError('\n'.join(errors))

    def gdrive_download_and_extract(file_id, archive_name, extract_to):
        archive_path = f'/tmp/{archive_name}'
        if not os.path.exists(archive_path):
            gdrive_download(file_id, archive_path)
        _extract(archive_path, extract_to)

    download_errors = []

    # ── 1. miniImageNet ──────────────────────────────────────────────────────
    DOWNLOAD_MINI = True
    if DOWNLOAD_MINI:
        mini_dst = os.path.join(DATASET_ROOT, 'mini-imagenet', 'compacted_datasets')
        if not os.path.exists(os.path.join(mini_dst, 'mini_imagenet_train.pickle')):
            try:
                gdrive_download_and_extract('12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D',
                                            'miniimagenet.archive', '/tmp/mini_raw')
                os.makedirs(mini_dst, exist_ok=True)
                for split in ['train', 'val', 'test']:
                    candidates = [os.path.join(r, f)
                                  for r, _, files in os.walk('/tmp/mini_raw')
                                  for f in files if split in f.lower() and f.endswith('.pickle')]
                    if candidates:
                        shutil.copy(candidates[0], os.path.join(mini_dst, f'mini_imagenet_{split}.pickle'))
                        print(f"  Placed {split} pickle")
                    else:
                        print(f"  WARNING: {split} pickle not found in archive")
            except Exception as e:
                msg = (f"miniImageNet: {e}\n"
                       f"  Manual: https://drive.google.com/file/d/12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D/view")
                print(f"ERROR: {msg}"); download_errors.append(msg)
        else:
            print("miniImageNet pickles already present.")

    # ── 2. tieredImageNet ────────────────────────────────────────────────────
    # All known public Google Drive links are no longer accessible (permission
    # denied — the sharing has been revoked, not a quota issue). Attempting them
    # here would generate several screens of error output for no benefit.
    # The dedicated recovery cell below handles this with learn2learn, requests,
    # and torchmeta — none of which depend on those Drive links.
    DOWNLOAD_TIERED = True
    if DOWNLOAD_TIERED:
        tiered_dst = os.path.join(DATASET_ROOT, 'tiered-imagenet')
        if not os.path.exists(os.path.join(tiered_dst, 'train_labels.pkl')):
            msg = ("tieredImageNet: Google Drive links are no longer publicly accessible.\n"
                   "  Run the tieredImageNet recovery cell (the cell immediately below).")
            print(f"\nINFO: {msg}"); download_errors.append(msg)
        else:
            print("tieredImageNet files already present.")

    # ── 3. CIFAR-FS ──────────────────────────────────────────────────────────
    DOWNLOAD_CIFAR = True
    if DOWNLOAD_CIFAR:
        cifar_dst = os.path.join(DATASET_ROOT, 'cifar_fs')
        if not os.path.exists(os.path.join(cifar_dst, 'train.pickle')):
            try:
                gdrive_download_and_extract('1GjGMI0q3bgcpcB_CjI40fX54WgLPuTpS',
                                            'cifar_fs.archive', '/tmp/cifar_raw')
                os.makedirs(cifar_dst, exist_ok=True)
                for split in ['train', 'val', 'test']:
                    candidates = [os.path.join(r, f)
                                  for r, _, files in os.walk('/tmp/cifar_raw')
                                  for f in files if f.endswith('.pickle') and split in f.lower()]
                    if candidates:
                        shutil.copy(candidates[0], os.path.join(cifar_dst, f'{split}.pickle'))
                        print(f"  Placed {split}.pickle")
                    else:
                        print(f"  WARNING: {split} pickle not found in CIFAR-FS archive")
            except Exception as e:
                msg = (f"CIFAR-FS: {e}\n"
                       f"  Manual: https://drive.google.com/file/d/1GjGMI0q3bgcpcB_CjI40fX54WgLPuTpS/view")
                print(f"ERROR: {msg}"); download_errors.append(msg)
        else:
            print("CIFAR-FS pickles already present.")

    # ── 4. CUB-200-2011 ─────────────────────────────────────────────────────
    # Archive layout (DeepEMD distribution, confirmed from DeepEMD cub.py source):
    #   cub/
    #     <11787 .jpg files — flat, no sub-folder>
    #     split/
    #       train.csv  val.csv  test.csv
    # CSV format: header row (skip), then col 0 = filename, col 1 = label/wnid.
    # Reference: github.com/icoz69/DeepEMD/blob/master/Models/dataloader/cub/fcn/cub.py
    DOWNLOAD_CUB = True
    if DOWNLOAD_CUB:
        cub_dst = os.path.join(DATASET_ROOT, 'cub-200-2011')
        if not os.path.exists(os.path.join(cub_dst, 'cub200_train.pickle')):
            try:
                gdrive_download_and_extract('1B8jmZin9teye7Lte9ZKsQ3lyMASbxune',
                                            'cub.archive', '/tmp/cub_raw')
                os.makedirs(cub_dst, exist_ok=True)

                # Strategy 1: archive already has .pickle files (some distributions)
                all_pickles = [os.path.join(r, f)
                               for r, _, files in os.walk('/tmp/cub_raw')
                               for f in files if f.endswith('.pickle')]
                split_map = {'train': 'cub200_train.pickle',
                             'val':   'cub200_val.pickle',
                             'test':  'cub200_test.pickle'}
                placed = 0
                for src in all_pickles:
                    for split, dst_name in split_map.items():
                        if split in os.path.basename(src).lower():
                            shutil.copy(src, os.path.join(cub_dst, dst_name))
                            print(f"  Copied {dst_name}"); placed += 1; break

                # Strategy 2: build pickles from the split/ CSV + flat image directory
                if placed < 3:
                    # Locate split/ dir — its parent directory holds the flat images
                    split_dir, images_root = None, None
                    for root, dirs, _ in os.walk('/tmp/cub_raw'):
                        if 'split' in dirs:
                            split_dir   = os.path.join(root, 'split')
                            images_root = root           # images are flat siblings of split/
                            break
                    if split_dir is None:                # fallback: CSVs at any depth
                        for root, _, files in os.walk('/tmp/cub_raw'):
                            if any(f in files for f in ('train.csv', 'val.csv', 'test.csv')):
                                split_dir   = root
                                images_root = os.path.dirname(root)
                                break

                    if images_root and split_dir:
                        print(f"  Images root : {images_root}")
                        print(f"  Split dir   : {split_dir}")
                        for split in ['train', 'val', 'test']:
                            csv_path = os.path.join(split_dir, f'{split}.csv')
                            if not os.path.exists(csv_path):
                                print(f"  WARNING: {csv_path} not found — skipping"); continue

                            # ── Read CSV using index-based parsing (matches DeepEMD's own cub.py)
                            # Header row is always skipped; col 0 = filename, col 1 = label.
                            with open(csv_path, 'r') as cf:
                                all_lines = cf.readlines()

                            # Print first 2 data lines for diagnostics (first line is header)
                            print(f"  CSV header : {all_lines[0].strip()}")
                            if len(all_lines) > 1:
                                print(f"  CSV row[1] : {all_lines[1].strip()}")

                            data_lines = [l.strip() for l in all_lines[1:] if l.strip()]
                            # DeepEMD drops broken image at training index 5864
                            if split == 'train' and len(data_lines) > 5864:
                                data_lines.pop(5864)

                            data = {}
                            missing = 0
                            for line in data_lines:
                                parts = line.split(',')
                                if len(parts) < 2:
                                    continue
                                fn    = parts[0].strip()   # filename (flat, no sub-dir)
                                label = parts[1].strip()   # wnid / class name
                                img_path = os.path.join(images_root, fn)
                                if not os.path.exists(img_path):
                                    # Some archives nest images one level deeper
                                    img_path = os.path.join(images_root,
                                                            os.path.basename(fn))
                                if os.path.exists(img_path):
                                    try:
                                        arr = np.array(
                                            pil_image.open(img_path)
                                            .convert('RGB').resize((84, 84)),
                                            dtype=np.uint8)
                                        data.setdefault(label, []).append(arr)
                                    except Exception:
                                        pass
                                else:
                                    missing += 1

                            if missing:
                                print(f"  WARNING: {missing} image files not found in archive")

                            dst_name = f'cub200_{split}.pickle'
                            with open(os.path.join(cub_dst, dst_name), 'wb') as pf:
                                pickle.dump(data, pf)
                            n_img = sum(len(v) for v in data.values())
                            print(f"  Built {dst_name} — {len(data)} classes, {n_img} images")
                    else:
                        raise RuntimeError(
                            "CUB: could not locate split/ directory in archive. "
                            "Print tree and inspect manually.")
            except Exception as e:
                msg = (f"CUB download/build failed: {e}\n"
                       f"  Manual: https://drive.google.com/file/d/1B8jmZin9teye7Lte9ZKsQ3lyMASbxune/view")
                print(f"ERROR: {msg}"); download_errors.append(msg)
        else:
            print("CUB pickles already present.")

    print("\nOption B complete. Run the verification cell below.")
    if download_errors:
        print("\n⚠ Some datasets require attention:")
        for err in download_errors:
            print(f"  • {err}")
else:
    print("Option A succeeded — Option B skipped.")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# tieredImageNet RECOVERY — run ONLY if the main download cell reported failure.
#
# All Google Drive links are revoked. This cell uses Zenodo (academic archive)
# as the primary source — no pip install needed: uses only requests, pickletools,
# numpy, and PIL (all pre-installed in Colab).
#
# Source: zenodo.org/record/7978538  (learn2learn's own mirror of Ren et al.)
# Format: _images_png.pkl (list of PNG bytes) + _labels.pkl (dict with
#         'label_specific' key). We decode PNGs → 84×84 uint8 numpy → NPZ.
#
# Download sizes: train ~7 GB, val ~2 GB, test ~3 GB (~12 GB total).
#
# MEMORY STRATEGY — guaranteed safe on 12 GB Colab RAM:
#   • pickletools.genops() streams the 7 GB pkl file ONE PNG AT A TIME — the
#     full 448 K-image list is NEVER loaded into Python RAM.
#   • Only bytes that begin with the PNG magic header are yielded; all other
#     bytes objects in the pickle stream (protocol metadata, dict keys, etc.)
#     are validated by magic bytes and silently skipped — no PIL errors.
#   • Each decoded image is written immediately to a disk-backed numpy memmap.
#   • NPZ is written by streaming the memmap in 4 MB chunks — no full array in RAM.
#   • HTTP Range headers resume an interrupted download from where it left off.
#   • A progress file resumes an interrupted decode without re-downloading.
# ─────────────────────────────────────────────────────────────────────────────
import os, io, pickle, shutil, gc
import pickletools          # stdlib — no pip needed
import requests
import numpy as np
from PIL import Image as pil_image

DATASET_ROOT = '/content/FSAKE/dataset'
TIERED_DEST  = os.path.join(DATASET_ROOT, 'tiered-imagenet')
ZENODO_BASE  = 'https://zenodo.org/record/7978538/files/tiered-imagenet-'

def tiered_done():
    return all(
        os.path.exists(os.path.join(TIERED_DEST, f'{s}_labels.pkl')) and
        os.path.exists(os.path.join(TIERED_DEST, f'{s}_images.npz'))
        for s in ['train', 'val', 'test'])

success = tiered_done()
if success:
    print("tieredImageNet already installed — nothing to do.")

# ── Approach A: Zenodo direct download ───────────────────────────────────────
if not success:
    try:
        os.makedirs(TIERED_DEST, exist_ok=True)

        # ── HTTP Range resume download ────────────────────────────────────────
        def _download_stream(url, dst_path, desc=''):
            existing = os.path.getsize(dst_path) if os.path.exists(dst_path) else 0
            headers  = {'Range': f'bytes={existing}-'} if existing else {}
            if existing:
                print(f"  {desc}: resuming from {existing >> 20} MB...")
            resp = requests.get(url, stream=True, timeout=60, headers=headers)
            if resp.status_code == 416:
                print(f"  {desc}: already complete."); return
            resp.raise_for_status()
            total      = int(resp.headers.get('content-length', 0)) + existing
            downloaded = existing
            next_report = ((existing >> 29) + 1) << 29
            with open(dst_path, 'ab' if existing else 'wb') as fh:
                for chunk in resp.iter_content(1 << 20):
                    fh.write(chunk)
                    downloaded += len(chunk)
                    if downloaded >= next_report:
                        pct = f'{downloaded*100//total}%' if total else f'{downloaded>>20} MB'
                        print(f"    {desc}: {pct} ({downloaded>>20} MB)")
                        next_report += 512 << 20
            print(f"  {desc}: done ({downloaded>>20} MB)")

        # ── Streaming PNG reader (memory-safe, no pickle.load) ────────────────
        # pickletools.genops() parses the pkl file one opcode at a time.
        # Only bytes objects that begin with the PNG magic header are yielded.
        # Any other bytes in the pickle stream (protocol bytes, dict keys, etc.)
        # are identified by magic and silently discarded — fixing the
        # "cannot identify image file" error from the previous version.
        _PNG_MAGIC  = b'\x89PNG\r\n\x1a\n'
        _BYTES_OPS  = frozenset(('SHORT_BINBYTES', 'BINBYTES', 'BINBYTES8', 'BYTEARRAY8'))

        def _stream_png_bytes(pkl_path, skip=0):
            """Yield valid PNG bytes one at a time from a pickle list-of-bytes file.
            Peak RAM = one PNG bytes object (~16-50 KB). Never calls pickle.load()."""
            count = 0
            if skip > 0:
                print(f"  Seeking past {skip} already-decoded images (sequential disk read)...")
            with open(pkl_path, 'rb') as fh:
                for opcode, arg, _ in pickletools.genops(fh):
                    if opcode.name not in _BYTES_OPS:
                        continue
                    b = bytes(arg)
                    # Validate PNG magic — skips metadata/protocol bytes silently
                    if len(b) < 8 or b[:8] != _PNG_MAGIC:
                        continue
                    if count < skip:
                        count += 1
                        continue
                    count += 1
                    yield b

        # ── NPZ writer: streams memmap in 4 MB chunks ─────────────────────────
        def _mmap_to_npz(npz_path, mmap_arr):
            import zipfile, numpy.lib.format as npformat
            with zipfile.ZipFile(npz_path, 'w',
                                 compression=zipfile.ZIP_DEFLATED,
                                 compresslevel=1) as zf:
                with zf.open('images.npy', 'w', force_zip64=True) as npy_f:
                    npformat.write_array(npy_f, mmap_arr, allow_pickle=False)

        # ── Per-split processing ──────────────────────────────────────────────
        for split in ['train', 'val', 'test']:
            print(f"\n── {split} ──")

            # Labels pkl is small (~3 MB) — safe to load with pickle.load()
            lbl_dst = os.path.join(TIERED_DEST, f'{split}_labels.pkl')
            if not os.path.exists(lbl_dst):
                _download_stream(ZENODO_BASE + f'{split}_labels.pkl',
                                 lbl_dst, f'{split}_labels.pkl')
            else:
                print(f"  {split}_labels.pkl already present.")

            npz_dst   = os.path.join(TIERED_DEST, f'{split}_images.npz')
            img_tmp   = os.path.join(TIERED_DEST, f'_tmp_{split}_images_png.pkl')
            mmap_path = os.path.join(TIERED_DEST, f'_tmp_{split}_images.dat')
            prog_path = os.path.join(TIERED_DEST, f'_tmp_{split}_progress.txt')

            if not os.path.exists(npz_dst):
                # Step 1: download PNG pkl (resumes if interrupted)
                _download_stream(ZENODO_BASE + f'{split}_images_png.pkl',
                                 img_tmp, f'{split}_images_png.pkl')

                # Step 2: get image count from the labels pkl (3 MB, safe)
                with open(lbl_dst, 'rb') as fh:
                    lbl_data = pickle.load(fh)
                n = len(lbl_data['label_specific'])
                del lbl_data; gc.collect()
                print(f"  {n} images to decode for {split}...")

                # Step 3: check for existing decode progress
                start_idx = 0
                if os.path.exists(mmap_path) and os.path.exists(prog_path):
                    try:
                        parts = open(prog_path).read().strip().split()
                        if len(parts) == 2 and int(parts[0]) == n:
                            start_idx = int(parts[1])
                    except Exception:
                        start_idx = 0
                if start_idx:
                    print(f"  Resuming decode from image {start_idx}/{n}...")

                # Step 4: disk-backed memmap — decoded array lives on disk, not in RAM
                mmap = np.memmap(mmap_path, dtype=np.uint8,
                                 mode='r+' if start_idx else 'w+',
                                 shape=(n, 84, 84, 3))

                # Step 5: stream PNG bytes from pkl → decode → write to memmap
                # Peak RAM per iteration: ~one PNG bytes object (~16-50 KB) +
                #                         one PIL image (21 KB) + one uint8 slice (21 KB)
                print(f"  Decoding (peak RAM ≈ <100 KB per image — streaming via pickletools)...")
                for i, png_bytes in enumerate(
                        _stream_png_bytes(img_tmp, skip=start_idx),
                        start=start_idx):
                    mmap[i] = np.asarray(
                        pil_image.open(io.BytesIO(png_bytes))
                        .convert('RGB')
                        .resize((84, 84), pil_image.BILINEAR),
                        dtype=np.uint8)
                    if (i + 1) % 10_000 == 0:
                        mmap.flush()
                        open(prog_path, 'w').write(f'{n} {i+1}')
                        print(f"    {i+1}/{n} decoded...")

                mmap.flush()
                print(f"  Writing {split}_images.npz (4 MB chunk streaming, minimal RAM)...")
                _mmap_to_npz(npz_dst, mmap)
                del mmap

                for p in [img_tmp, mmap_path, prog_path]:
                    if os.path.exists(p): os.remove(p)
                print(f"  {split}_images.npz saved.")
            else:
                print(f"  {split}_images.npz already present.")
                for p in [img_tmp, mmap_path, prog_path]:
                    if os.path.exists(p): os.remove(p)

        success = tiered_done()
        if success:
            print("\nApproach A (Zenodo) succeeded — tieredImageNet is ready.")
    except Exception as e:
        import traceback
        print(f"\n  [A] Failed: {e}")
        traceback.print_exc()

# ── Approach B: Drive (last-ditch, links likely still revoked) ────────────────
if not success:
    TIERED_IDS = [
        '1ANczVwnI1BDHIF65TgulaGALFnXBvRfs',
        '1nVGCTd9ttULRXFezh4xILQ9lUkg0WZCG',
        '1g1aIDy2Ar_MViF2gDXFYDBTR-HYecV07',
    ]
    import tarfile, zipfile
    for tid in TIERED_IDS:
        arc = '/tmp/tiered_b.archive'
        raw = '/tmp/tiered_b_raw'
        try:
            sess  = requests.Session()
            url   = f'https://docs.google.com/uc?export=download&id={tid}'
            r     = sess.get(url, stream=True, timeout=30)
            token = next((v for k, v in r.cookies.items()
                          if k.startswith('download_warning')), None)
            if token:
                r = sess.get(url + f'&confirm={token}', stream=True, timeout=600)
            with open(arc, 'wb') as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
            size_mb = os.path.getsize(arc) >> 20
            print(f"  [B] {size_mb} MB downloaded")
            if size_mb < 100:
                raise RuntimeError(f"File too small ({size_mb} MB) — Drive link still blocked")
            os.makedirs(raw, exist_ok=True)
            with open(arc, 'rb') as fh:
                magic = fh.read(4)
            if magic[:2] == b'PK':
                with zipfile.ZipFile(arc) as z: z.extractall(raw)
            else:
                with tarfile.open(arc) as t: t.extractall(raw, filter='data')
            os.makedirs(TIERED_DEST, exist_ok=True)
            for ro, _, files in os.walk(raw):
                for f in files:
                    if f.endswith(('.pkl', '.npz')):
                        shutil.copy(os.path.join(ro, f), os.path.join(TIERED_DEST, f))
                        print(f"  [B] Placed {f}")
            success = tiered_done()
            if success:
                print("  Approach B (Drive) succeeded."); break
        except Exception as e:
            print(f"  [B] {tid}: {e}")
        finally:
            for p in [arc, raw]:
                if os.path.exists(p):
                    (os.remove if os.path.isfile(p) else shutil.rmtree)(p)

# ── Manual fallback ───────────────────────────────────────────────────────────
if not success:
    print("""
╔══════════════════════════════════════════════════════════════════╗
║  tieredImageNet — all automatic methods failed                   ║
╠══════════════════════════════════════════════════════════════════╣
║  Re-running this cell resumes safely:                            ║
║    • HTTP Range header resumes a partial download                ║
║    • Progress file resumes the decode from the last checkpoint   ║
║                                                                  ║
║  If Zenodo is unreachable, download manually:                    ║
║    https://zenodo.org/record/7978538                             ║
║  Required files (all 6):                                         ║
║    tiered-imagenet-{train,val,test}_labels.pkl                   ║
║    tiered-imagenet-{train,val,test}_images_png.pkl               ║
║                                                                  ║
║  Upload to /content/FSAKE/dataset/tiered-imagenet/ then re-run. ║
╚══════════════════════════════════════════════════════════════════╝
""")
else:
    print("tieredImageNet is ready.")


### Verify Dataset Folder Structure

Expected layout inside `/content/FSAKE/dataset/`:

```
dataset/
  mini-imagenet/
    compacted_datasets/
      mini_imagenet_train.pickle
      mini_imagenet_val.pickle
      mini_imagenet_test.pickle
  tiered-imagenet/
      train_labels.pkl  |  train_images.npz
      val_labels.pkl    |  val_images.npz
      test_labels.pkl   |  test_images.npz
  cub-200-2011/
      cub200_train.pickle
      cub200_val.pickle
      cub200_test.pickle
  cifar_fs/
      train.pickle
      val.pickle
      test.pickle
```

In [ ]:
import os

dataset_root = '/content/FSAKE/dataset'

# Expected sub-folders and key files per dataset
expected = {
    'mini-imagenet/compacted_datasets': [
        'mini_imagenet_train.pickle', 'mini_imagenet_val.pickle', 'mini_imagenet_test.pickle'
    ],
    'tiered-imagenet': [
        'train_labels.pkl', 'val_labels.pkl', 'test_labels.pkl'
    ],
    'cub-200-2011': [
        'cub200_train.pickle', 'cub200_val.pickle', 'cub200_test.pickle'
    ],
    'cifar_fs': [
        'train.pickle', 'val.pickle', 'test.pickle'
    ],
}

all_ok = True
for sub, files in expected.items():
    folder = os.path.join(dataset_root, sub)
    if not os.path.exists(folder):
        print(f"MISSING folder: {folder}")
        all_ok = False
        continue
    for f in files:
        path = os.path.join(folder, f)
        status = "OK" if os.path.exists(path) else "MISSING"
        if status == "MISSING":
            all_ok = False
        print(f"  [{status}] {sub}/{f}")

if all_ok:
    print("\nAll dataset files found. Ready to train.")

## Step 4: Configure Dataset Paths

Inspect the repo's config/argument files and patch the data directory path.

In [ ]:
import os
os.chdir('/content/FSAKE')

# Confirm dataset_root is 'dataset' (relative to repo root) in train.py
print("--- Dataset root setting in train.py ---")
!grep -n "dataset_root" /content/FSAKE/train.py | head -5

# Confirm data.py backslash bug is patched (should show forward-slash-safe path)
print("\n--- mini-imagenet path construction in data.py (should use separate 'mini-imagenet', 'compacted_datasets' args) ---")
!grep -n "mini-imagenet" /content/FSAKE/data.py | head -3

print("\n--- Dataset name mapping ---")
print("  'mini'   -> dataset/mini-imagenet/compacted_datasets/mini_imagenet_<split>.pickle")
print("  'tiered' -> dataset/tiered-imagenet/<split>_labels.pkl + <split>_images.npz")
print("  'cub'    -> dataset/cub-200-2011/cub200_<split>.pickle")
print("  'cifar'  -> dataset/cifar_fs/<split>.pickle")


In [ ]:
# dataset_root is 'dataset' (a relative path) — resolved from wherever you run train.py.
# The %cd /content/FSAKE in Step 5 ensures 'dataset' resolves correctly.
#
# If your pickle files are stored somewhere else, you can override the root:

# import re
# new_root = '/content/drive/MyDrive/FSAKE_datasets'   # ← your actual path
# for script in ['/content/FSAKE/train.py', '/content/FSAKE/eval.py']:
#     txt = open(script).read()
#     txt = re.sub(r"tt\.arg\.dataset_root\s*=\s*'dataset'",
#                  f"tt.arg.dataset_root = '{new_root}'", txt)
#     open(script, 'w').write(txt)
#     print(f"Patched dataset_root in {script}")

print("dataset_root = 'dataset' (relative to /content/FSAKE)")
print("Keep %cd /content/FSAKE before any train.py / eval.py call.")


## Step 5: Run Training

✓ Checkpoints auto-save to Hugging Face during training.  
✓ Run same command again to resume from checkpoint.

**Training argument reference:**
- `--dataset` : `mini` | `tiered` | `cub` | `cifar`
- `--num_ways` : `5` (standard)
- `--num_shots` : `1` or `5` (few-shot settings)
- `--pool_mode` : `support` (default) or `kn`
- `--transductive` : `True` (default) or `False`


## Step 5A: Quick Setup — Checkpoint Storage (Hugging Face ← Default)

**Hugging Face Hub is auto-configured below. Just run the cell and authenticate when prompted.**



In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# AUTOMATIC SETUP: Hugging Face Hub for Checkpoint Storage (Default)
# ═════════════════════════════════════════════════════════════════════════════
# 
# This cell automatically:
# 1. Installs huggingface_hub
# 2. Authenticates (if needed)
# 3. Creates repo for storing checkpoints
# 4. Sets up checkpoint directory
#
# No configuration needed — just run this cell once! 🚀

import os
import subprocess
import sys
from pathlib import Path

print("🚀 Setting up Hugging Face Hub for automatic checkpoint storage...\n")

# Install huggingface_hub
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

from huggingface_hub import HfApi, create_repo, whoami

# Try to get current HF identity
try:
    user_info = whoami()
    HF_USERNAME = user_info['name']
    print(f"✓ Already logged in to Hugging Face as: {HF_USERNAME}\n")
except Exception as e:
    print("🔐 Not authenticated yet. Logging in...\n")
    # Use huggingface_hub login (stores token in ~/.huggingface/token)
    # Use huggingface_hub login with credentials
    HF_TOKEN = 'YOUR_HF_TOKEN_HERE'  # <-- Replace with your token
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=True)
    
    try:
        user_info = whoami()
        HF_USERNAME = user_info['name']
        print(f"\n✓ Logged in as: {HF_USERNAME}\n")
    except:
        print("\n⚠️  Authentication failed. Login manually ")
        HF_USERNAME = None

if HF_USERNAME:
    HF_REPO_NAME = "fsake-checkpoints"
    HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
    
    # Create repo (this is idempotent — safe to run multiple times)
    try:
        api = HfApi()
        api.repo_info(HF_REPO_ID)
        print(f"✓ Repository exists: https://huggingface.co/{HF_REPO_ID}")
    except:
        try:
            print(f"Creating repository: {HF_REPO_ID}...")
            create_repo(HF_REPO_ID, private=False, exist_ok=True)
            print(f"✓ Repository created: https://huggingface.co/{HF_REPO_ID}")
        except Exception as e:
            print(f"Note: Repository creation: {e}")
    
    # Setup checkpoint directory (symlink to local storage)
    CHECKPOINT_DIR = '/tmp/fsake_hf_checkpoints'
    LOCAL_CHECKPOINT_DIR = '/content/FSAKE/asset/checkpoints'
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    
    # Remove old symlink if exists
    if os.path.islink(LOCAL_CHECKPOINT_DIR):
        os.remove(LOCAL_CHECKPOINT_DIR)
    elif os.path.isdir(LOCAL_CHECKPOINT_DIR) and not os.listdir(LOCAL_CHECKPOINT_DIR):
        os.rmdir(LOCAL_CHECKPOINT_DIR)
    
    # Create symlink
    if not os.path.exists(LOCAL_CHECKPOINT_DIR):
        os.makedirs(os.path.dirname(LOCAL_CHECKPOINT_DIR), exist_ok=True)
        os.symlink(CHECKPOINT_DIR, LOCAL_CHECKPOINT_DIR)
    
    print(f"\n✓ Checkpoint storage ready!")
    print(f"  • Saving to: Hugging Face Hub")
    print(f"  • Repository: {HF_REPO_ID}")
    print(f"  • Local staging: {CHECKPOINT_DIR}")
    print(f"\n💡 After training completes:")
    print(f"  1. Run 'Push Checkpoints to Hugging Face' cell to upload")
    print(f"  2. Share repo URL with collaborators")
    print(f"  3. Others can download anytime without authentication")
    
    # Store for later use
    import json
    config_file = '/tmp/fsake_hf_config.json'
    with open(config_file, 'w') as f:
        json.dump({'HF_USERNAME': HF_USERNAME, 'HF_REPO_ID': HF_REPO_ID}, f)
else:
    print("\n⚠️  Setup incomplete. Please troubleshoot:")
    print("  • Check your Hugging Face token at: https://huggingface.co/settings/tokens")


In [ ]:
%cd /content/FSAKE
!python train.py --dataset mini --num_ways 5 --num_shots 1 \
    --transductive True --pool_mode support --unet_mode addold


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# AUTO PUSH: Checkpoints to Hugging Face Hub
# ═════════════════════════════════════════════════════════════════════════════
# 
# Run this after training to upload all checkpoints to your Hugging Face repo.
# This cell automatically reads the credentials set up in Step 5A.

import os
import json
from datetime import datetime

# Load config from Step 5A
config_file = '/tmp/fsake_hf_config.json'
if not os.path.exists(config_file):
    print("ERROR: Setup not completed. Run Step 5A cell first.")
else:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    HF_USERNAME = config['HF_USERNAME']
    HF_REPO_ID = config['HF_REPO_ID']
    CHECKPOINT_DIR = '/tmp/fsake_hf_checkpoints'
    
    print(f"Pushing checkpoints to Hugging Face Hub...\n")
    
    if not os.path.exists(CHECKPOINT_DIR) or not os.listdir(CHECKPOINT_DIR):
        print("ERROR: No checkpoints found to upload.")
        print(f"  Expected location: {CHECKPOINT_DIR}")
        print(f"  Did you run training yet?")
    else:
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            
            print(f"Repository: https://huggingface.co/{HF_REPO_ID}\n")
            print("Uploading experiments...\n")
            
            total_experiments = 0
            total_files = 0
            total_size_mb = 0
            
            # Upload each experiment folder
            for exp_name in sorted(os.listdir(CHECKPOINT_DIR)):
                exp_path = os.path.join(CHECKPOINT_DIR, exp_name)
                if not os.path.isdir(exp_path):
                    continue
                
                total_experiments += 1
                print(f"📦 {exp_name}")
                
                # List files to upload
                for fname in sorted(os.listdir(exp_path)):
                    fpath = os.path.join(exp_path, fname)
                    if os.path.isfile(fpath):
                        size_mb = os.path.getsize(fpath) / (1024*1024)
                        total_size_mb += size_mb
                        total_files += 1
                        
                        print(f"   ├─ {fname:<25} ({size_mb:6.1f} MB)...", end=' ', flush=True)
                        
                        try:
                            api.upload_file(
                                path_or_fileobj=fpath,
                                path_in_repo=f"{exp_name}/{fname}",
                                repo_id=HF_REPO_ID,
                                repo_type="model"
                            )
                            print("✓")
                        except Exception as e:
                            print(f"✗ ({str(e)[:30]})")
            
            print(f"\n{'='*70}")
            print(f"✓ Upload complete!")
            print(f"  • Experiments: {total_experiments}")
            print(f"  • Files: {total_files}")
            print(f"  • Size: {total_size_mb:.1f} MB")
            print(f"  • Repository: https://huggingface.co/{HF_REPO_ID}")
            print(f"{'='*70}")
            
            print(f"\n📋 Share with collaborators:")
            print(f"\nCode for others to download:")
            print(f"""
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download('{HF_REPO_ID}', 'experiment_name/model_best.pth.tar')
""")
            
        except Exception as e:
            print(f"ERROR during upload: {e}")


In [ ]:
%cd /content/FSAKE

# miniImageNet — 5-way 5-shot (transductive, kn pooling)
!python train.py --dataset mini --num_ways 5 --num_shots 5 \
    --transductive True --pool_mode kn --unet_mode addold

In [ ]:
%cd /content/FSAKE

# CIFAR-FS — 5-way 1-shot
!python train.py --dataset cifar --num_ways 5 --num_shots 1 \
    --transductive True --pool_mode support --unet_mode addold

# Other datasets — uncomment as needed:
# !python train.py --dataset tiered --num_ways 5 --num_shots 1 --transductive True --pool_mode support --unet_mode addold
# !python train.py --dataset cub    --num_ways 5 --num_shots 1 --transductive True --pool_mode support --unet_mode addold

In [ ]:
import os
from datetime import datetime

print("Checkpoint Status:\n")

CKPT_DIR = '/tmp/fsake_hf_checkpoints'

if not os.path.exists(CKPT_DIR) or not os.listdir(CKPT_DIR):
    print("No checkpoints yet. Run training to create them.")
else:
    experiments = [d for d in os.listdir(CKPT_DIR) if os.path.isdir(os.path.join(CKPT_DIR, d))]
    
    for exp in sorted(experiments)[-5:]:  # Show last 5
        exp_path = os.path.join(CKPT_DIR, exp)
        best_file = os.path.join(exp_path, 'model_best.pth.tar')
        
        if os.path.exists(best_file):
            mod_time = os.path.getmtime(best_file)
            time_str = datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
            size_mb = os.path.getsize(best_file) / (1024 * 1024)
            print(f"✓ {exp}")
            print(f"  {time_str} | {size_mb:.1f} MB\n")

print("Tip: Re-run training to resume from model_best.pth.tar")


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Download Checkpoints from Another User's Hugging Face Repo
# ═════════════════════════════════════════════════════════════════════════════
# Use this if someone else has already trained FSAKE and shared their checkpoint URL.
# No authentication needed!

import os
import shutil

HF_REPO_ID = "username/fsake-checkpoints"  # ← CHANGE: Other user's repo
EXP_NAME = "D-mini_N-5_K-1_Q-5_B-40_..."   # ← CHANGE: Experiment name from their repo

print(f"Downloading checkpoints from: {HF_REPO_ID}/{EXP_NAME}\n")

from huggingface_hub import hf_hub_download

CKPT_DIR = '/content/FSAKE/asset/checkpoints'
os.makedirs(os.path.join(CKPT_DIR, EXP_NAME), exist_ok=True)

for filename in ['model_best.pth.tar', 'best_log.pth.tar']:
    try:
        print(f"Downloading {filename}...", end=' ', flush=True)
        path = hf_hub_download(HF_REPO_ID, f'{EXP_NAME}/{filename}')
        dst = os.path.join(CKPT_DIR, EXP_NAME, filename)
        shutil.copy(path, dst)
        size_mb = os.path.getsize(dst) / (1024*1024)
        print(f"✓ ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"(not found)")

print("\n✓ Ready to resume. Run training with same arguments:")
print(f"  python train.py --dataset mini --num_ways 5 --num_shots 1 ...")


## Step 6: Run Evaluation

After training, evaluate the model on the test split.

In [ ]:
%cd /content/FSAKE

# eval.py is the dedicated evaluation script (confirmed in repo).
# It loads the best checkpoint from asset/checkpoints/<experiment_name>/model_best.pth.tar
# Arguments must EXACTLY match the training run so the experiment name resolves correctly.

# Evaluate miniImageNet 5-way 1-shot
!python eval.py --dataset mini --num_ways 5 --num_shots 1 \
    --transductive True --pool_mode support --unet_mode addold

# Evaluate miniImageNet 5-way 5-shot (uncomment if you trained this config)
# !python eval.py --dataset mini --num_ways 5 --num_shots 5 \
#     --transductive True --pool_mode kn --unet_mode addold

## Step 7: Verify & Backup Results

This cell checks your Hugging Face checkpoints and provides next steps.

**Next:** Run the "Push Checkpoints to Hugging Face" cell after this.


In [ ]:
import os
from datetime import datetime

print("🎯 Training Complete!\n")
print("="*70)

# Check for checkpoints
CKPT_DIR = '/tmp/fsake_hf_checkpoints'

if os.path.exists(CKPT_DIR) and os.listdir(CKPT_DIR):
    experiments = [d for d in os.listdir(CKPT_DIR) if os.path.isdir(os.path.join(CKPT_DIR, d))]
    print(f"✓ Checkpoints ready ({len(experiments)} experiment(s))")
    
    print("\nNext steps:")
    print("  1. Run 'Push Checkpoints to Hugging Face' cell")
    print("  2. Share Hugging Face repo URL with collaborators")
    print("  3. Others can download with hf_hub_download() anytime")
else:
    print("⚠️  No checkpoints found")
    print("   Did training run? Check logs above.")

print("="*70)


## Troubleshooting

### Common Issues and Fixes

In [ ]:
# Full diagnostics — run any time to check environment state
import torch, sys, os

print("=" * 55)
print(f"Python:           {sys.version.split()[0]}")
print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"VRAM:             {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n--- Key package versions ---")
!pip show torch torch-geometric opencv-python tensorboardx 2>/dev/null \
    | grep -E "^Name:|^Version:"

print("\n--- FSAKE repo files ---")
if os.path.exists('/content/FSAKE'):
    !ls /content/FSAKE
else:
    print("Repository not cloned yet.")

print("\n--- Checkpoint directory ---")
ckpt = '/content/FSAKE/asset/checkpoints'
if os.path.exists(ckpt):
    !ls "{ckpt}"
else:
    print(f"{ckpt} not found (no training completed yet).")

In [ ]:
# Fix: CUDA not available (wrong runtime type)
import torch
if not torch.cuda.is_available():
    print("ERROR: CUDA not available.")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU → Save")
else:
    print("GPU is ready.")

In [ ]:
# Fix: Missing module errors
# Run this cell only if you get "ModuleNotFoundError"

!pip install torch_geometric
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu118.html

In [ ]:
# Fix: Dataset path errors
# Run if you get FileNotFoundError about pickle files

import os

# Check all four datasets
checks = {
    'mini':   'dataset/mini-imagenet/compacted_datasets/mini_imagenet_train.pickle',
    'tiered': 'dataset/tiered-imagenet/train_labels.pkl',
    'cub':    'dataset/cub-200-2011/cub200_train.pickle',
    'cifar':  'dataset/cifar_fs/train.pickle',
}

os.chdir('/content/FSAKE')
for name, rel_path in checks.items():
    full = os.path.join('/content/FSAKE', rel_path)
    status = "FOUND" if os.path.exists(full) else "MISSING"
    print(f"[{status}] --dataset {name}  -->  {rel_path}")

print("\nIf any are MISSING: upload the pickle files to Google Drive and re-run Step 3.")

## Expected Results

From the paper (FSAKE, Expert Systems with Applications):

| Dataset | 5-way 1-shot | 5-way 5-shot |
|---------|-------------|-------------|
| miniImageNet | 61.86% | 79.07% |
| CIFAR-FS | 74.12% | 87.10% |
| tieredImageNet | 68.04% | 82.99% |
| CUB-200-2011 | 76.48% | 89.18% |

Training logs print accuracy at each validation step (every 400 iterations):
```
evaluation: total_count=..., accuracy: mean=61.86%, std=0.46%, ci95=0.18%
```

---

**Key notes:**
- FSAKE uses **episodic (meta-learning) training** — each iteration is one few-shot episode.
- Checkpoints are saved to `asset/checkpoints/<experiment_name>/model_best.pth.tar`.
- `eval.py` must be called with the **same arguments** as `train.py` to load the right checkpoint.
- Total training: 100 000 iterations (mini/cifar/cub) or 200 000 iterations (tiered).

## Checkpoint Sharing Across Google Accounts

This is a summary of the three options for saving and sharing checkpoints so that training can be resumed by any user (even with a different Google account).

### Quick Comparison

| Option | Setup Time | Share Method | Who Can Access | Best For |
|--------|-----------|--------------|-----------------|----------|
| **1. Hugging Face Hub** | 5 min | Public URL | Anyone (no auth needed) | **Recommended** — easiest, most flexible |
| **2. Google Cloud Storage** | 20 min | Public URL or API | Anyone with URL | Large-scale collaboration or production |
| **3. Google Drive Shared** | 2 min | Shared folder link | Google account holders only | Quick team sharing |

### Workflow Examples

**Scenario A: User 1 trains, User 2 resumes (Different Google Accounts)**

| Step | Hugging Face | Google Drive | GCS |
|------|-----|-----|-----|
| 1. Train | Run HF cell, then train | Run GDrive cell, then train | Run GCS cell, then train |
| 2. Save | Auto-save to HF repo | Auto-save to GDrive folder | Auto-save to GCS bucket |
| 3. Upload | Run "Push to HF" cell | Auto-synced (nothing to do) | Auto-synced (nothing to do) |
| 4. Share | Send HF URL to User 2 | Send GDrive link to User 2 | Send GCS link to User 2 |
| 5. User 2 resumes | Run "Download from HF" cell | Mount their own Drive + symlink | Download from GCS URL |

### Step-by-Step for Hugging Face (Recommended)

1. **Setup (once per account):**
   - Go to [huggingface.co/join](https://huggingface.co/join) → Create account
   - Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → Create token
   - Run Option 1 cell, paste token when prompted

2. **After training:**
   - Run "Push Checkpoints to Hugging Face" cell
   - Share the repo URL with other users

3. **Other users download:**
   - Run "Download Checkpoints from Another User" cell
   - Select Option A (Hugging Face)
   - Paste the repo ID and experiment name

### Resuming After Download

Once checkpoints are downloaded/mounted in `/content/FSAKE/asset/checkpoints/`, simply run the same training command:

```bash
python train.py --dataset mini --num_ways 5 --num_shots 1 \
    --transductive True --pool_mode support --unet_mode addold
```

The model will automatically detect and load `model_best.pth.tar` and resume from where it left off.
